# Crivo — treinar o analisador de frases em GPU

Este notebook treina, **do zero**, o analisador de frases do Crivo: para cada palavra ele aprende
a **classe** (substantivo, verbo, artigo…), o **lema** ("abriu" → "abrir") e **de qual palavra ela depende**
e com que ligação (sujeito, objeto, conectivo…), como no exemplo:

> *A menina abriu o guarda-chuva porque começou a chover.*
> "menina" é sujeito de "abriu"; "guarda-chuva" é objeto; "porque … chover" é a causa.

**Modelo:** BiLSTM 2×200 com pontuação biafim (Dozat & Manning, 2017), em PyTorch. Os pesos são
exportados para NumPy: o Crivo usa o modelo **sem** torch.

**Dados:** [UD Portuguese-Bosque](https://github.com/UniversalDependencies/UD_Portuguese-Bosque)
(licença **CC BY-SA 4.0**), versão fixada por commit e conferida por SHA-256. Os pesos gerados
seguem a mesma licença, com crédito ao Universal Dependencies / Bosque (Floresta Sintá(c)tica, Linguateca).

### Antes de começar
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**.
2. Rode as células em ordem (**Ambiente de execução → Executar tudo**). Leva uns 15–25 minutos.
3. No fim, baixe `analisador_pt.zip` e me envie (ou suba no GitHub, como explicado no final).

In [ ]:
# 1. Conferir a GPU
!nvidia-smi -L || echo "Sem GPU: ative em Ambiente de execução → Alterar o tipo de ambiente de execução"
import torch
print("torch", torch.__version__, "| GPU disponível:", torch.cuda.is_available())

In [ ]:
# 2. Baixar o código do Crivo
# Enquanto o analisador não estiver no main, use o branch de desenvolvimento.
BRANCH = "claude/determined-heisenberg-moldn2"  #@param {type:"string"}
!rm -rf CRIVO && git clone --depth 1 -b "$BRANCH" https://github.com/HROSONE/CRIVO.git
%cd CRIVO
!git log --oneline -1

In [ ]:
# 3. Baixar o UD Portuguese-Bosque na versão fixada (o treino confere o SHA-256)
import json, urllib.request, pathlib
origem = json.loads(pathlib.Path("dados/origem_ud_bosque.json").read_text(encoding="utf-8"))
pasta = pathlib.Path("ud"); pasta.mkdir(exist_ok=True)
for nome in origem["sha256"]:
    urllib.request.urlretrieve(origem["url_base"] + nome, pasta / nome)
    print("baixado", nome)
print("Licença:", origem["licenca"], "-", origem["credito"])

In [ ]:
# 4. Treinar (cada época mostra a nota no conjunto de validação)
EPOCAS = 40  #@param {type:"integer"}
!python scripts/treinar_analisador_biafim.py --dados ud --saida artefatos/analisador_pt --epocas $EPOCAS

In [ ]:
# 5. Ver o resultado: notas no teste oficial e uma análise de exemplo
import json, importlib, analisador_frases as af
importlib.reload(af)
meta = json.loads(open("artefatos/analisador_pt/meta.json", encoding="utf-8").read())
print("Teste oficial:", meta["avaliacao"]["teste"])
print("Mínimo exigido:", af.MINIMO)
a = af.AnalisadorFrases()
print("Analisador ligado:", a.disponivel, a.motivo)
for frase in ["A menina abriu o guarda-chuva porque começou a chover.",
              "Meu cachorro latiu a noite toda porque viu um gato.",
              "Tô cansado do trabalho."]:
    print("\n" + frase)
    print(af.explicar(frase, a))

In [ ]:
# 6. Baixar os pesos (modelo.npz + meta.json)
!cd artefatos && zip -r ../analisador_pt.zip analisador_pt && cd .. && ls -la analisador_pt.zip
from google.colab import files
files.download("analisador_pt.zip")

### Depois do treino
Me envie o `analisador_pt.zip`, ou suba os dois arquivos no GitHub:
abra o branch indicado acima, entre em `artefatos/analisador_pt/` (crie a pasta se não existir),
**Add file → Upload files**, envie `modelo.npz` e `meta.json` e faça o commit.

Eu rodo os testes e as baterias com o modelo novo antes de ligar no Crivo. O analisador só é
usado se as notas no teste oficial passarem do mínimo (`af.MINIMO`).